### 1. Prepare pdbqt for 8vb5 and a1aac(as native ligand)

In [24]:
import os
import re
import glob
import csv
import sys
import numpy as np
import pandas as pd
import urllib.request
from pathlib import Path
# from Bio.PDB import MMCIFParser, PDBParser, PDBIO, Select

In [60]:
BASE_DIR = Path("/home/ssm-user/project/autodock")
BASE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(BASE_DIR)
print("BASE_DIR is set to:", BASE_DIR)

BASE_DIR is set to: /home/ssm-user/project/autodock


In [15]:
# pdb_id = "8vb5"
# cif = f"{pdb_id.upper()}.cif"
# pdb = f"{pdb_id.lower()}.pdb"

# urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id.upper()}.cif", str(cif))
# parser = MMCIFParser()
# structure = parser.get_structure(pdb_id.upper(), str(cif))
# io = PDBIO()
# io.set_structure(structure)
# io.save(str(pdb))

# print("Saved PDB:", pdb)

Saved PDB: 8vb5.pdb


In [ ]:
# # in bash
# wget https://sw-tools.rcsb.org/apps/MAXIT/maxit-v11.300-prod-src.tar.gz
# tar -xvzf maxit-v11.300-prod-src.tar.gz
# maxit -input /home/ssm-user/project/autodock/8VB5.cif -output /home/ssm-user/project/autodock/8vb5.pdb -o PDB

In [3]:
residues = set()

with open('8vb5.pdb', 'r') as f:
    for l in f:
        if l.startswith('HETATM'):
            residues.add((l[17:22], l[21:24]))

for resname, chain in residues:
    print(resname, chain)

HOH A A13
HOH A A14
HOH A A12
EDO A A11
AAC A A11
PEG A A11
 CL A A11


In [4]:
protein_file = "8vb5.pdb"
output_protein_file = "8vb5_prot.pdb"
ligand_file = "a1aac.pdb"
# fixed_ligand_file = "a1aac_fixed.pdb"

In [5]:
# ------------------------
# Step 1: Prepare protein for pdb2pqr
# ------------------------
with open(output_protein_file, 'w') as g:
    with open(protein_file, 'r') as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                resname = line[17:20].strip()
                if line.startswith('ATOM') and line[21] == 'A':

                    res_num_str = line[22:26].strip()
                    res_num_pure = ''.join(filter(str.isdigit, res_num_str))

                    new_line = list(line)
                    new_line[22:26] = f"{res_num_pure:>4}"
                    g.write("".join(new_line))
                elif line.startswith('HETATM') and resname not in ['EDO', 'PEG']:
                    g.write(line)
            elif line.startswith('TER'):
                g.write('TER\n')
        g.write('END\n')
print(f"Protein file created: {output_protein_file}")

Protein file created: 8vb5_prot.pdb


In [6]:
# ------------------------
# Step 2: Extract ligand (A1AAC)
# ------------------------
with open(os.path.join(BASE_DIR, "a1aac.pdb"), "w") as g:
    with open(protein_file, 'r') as f:
        for line in f:
            if line.startswith('HETATM') and line[17:20].strip() == "AAC":
                g.write(line)
        g.write("END\n")

# ------------------------
# Step 3: Fix ligand PDB for AutoDock
# ------------------------
# with open(fixed_ligand_file, "w") as out_file:
#     with open(ligand_file, "r") as in_file:
#         for line in in_file:
#             if line.startswith("HETATM"):
#                 fixed_line = list(line)
#                 fixed_line[17:20] = ['A','A','C']
#                 fixed_line[21] = 'A'
#                 fixed_line[22:26] = list(f"{1:>4}")
#                 atom_name = line[12:16].strip()
#                 if 'C' in atom_name:
#                     fixed_line[76:78] = [' ', 'C']
#                 elif 'N' in atom_name:
#                     fixed_line[76:78] = [' ', 'N']
#                 elif 'O' in atom_name:
#                     fixed_line[76:78] = [' ', 'O']
#                 fixed_line[78:80] = [' ', ' ']
#                 out_file.write("".join(fixed_line))
#             else:
#                 out_file.write(line)
# print(f"Fixed ligand file created: {fixed_ligand_file}")


In [7]:
# ------------------------
# Step 4: Calculate geometric center of ligand
# ------------------------
ligand_geom = []
coord_pattern = re.compile(r'HETATM\s+.*?\s+([-+]?\d*\.\d+)\s+([-+]?\d*\.\d+)\s+([-+]?\d*\.\d+)')
with open(ligand_file, 'r') as f:
    for line in f:
        m = coord_pattern.match(line)
        if m:
            x, y, z = [float(c) for c in m.groups()]
            ligand_geom.append([x, y, z])

if ligand_geom:
    ligand_geom = np.array(ligand_geom)
    center = ligand_geom.mean(axis=0)
    print(f"Geometric Center of ligand: {center[0]:.3f} {center[1]:.3f} {center[2]:.3f}")
else:
    print("Warning: No coordinates could be extracted. Check the input file format.")

Geometric Center of ligand: -1.128 6.733 -18.115


In [ ]:
# pdb2pqr: AMBER99ff parameter (pH 7.4)
! /home/ssm-user/miniforge3/envs/docking/bin/pdb2pqr30 \
    --ff AMBER \
    --keep-chain \
    --titration-state-method propka \
    --with-ph 7.4 \
    8vb5_prot.pdb \
    8vb5_prot.pqr

In [9]:
# pqr -> pdbqt
! /home/ssm-user/apps/mgltools/bin/pythonsh \
    /home/ssm-user/apps/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_receptor4.py \
    -r 8vb5_prot.pqr \
    -o 8vb5_prot.pdbqt \
    -C \
    -U nphs_lps \
    -v

setting PYTHONHOME environment
set verbose to  True
read  8vb5_prot.pqr
setting up RPO with mode= automatic and outputfilename=  8vb5_prot.pdbqt
charges_to_add= None
delete_single_nonstd_residues= None


In [10]:
# native ligand: pdb -> mol2
! obabel -ipdb a1aac.pdb -omol2 -O a1aac.mol2

1 molecule converted


In [11]:
# mol2 -> pdbqt
! /home/ssm-user/apps/mgltools/bin/pythonsh \
    /home/ssm-user/apps/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py \
    -l a1aac.mol2 \
    -o a1aac.pdbqt \
    -U nphs_lps \
    -v

setting PYTHONHOME environment
set verbose to  True
read  a1aac.mol2
setting up LPO with mode= automatic and outputfilename=  a1aac.pdbqt
and check_for_fragments= False
and bonds_to_inactivate= 
returning  0
No change in atomic coordinates


### 2. Native ligand(a1aac) based AutoGrid
---
all possible atom types should be specified for virtual screening since we will handle diverse molecules.

AutoDock Atom types:
https://autodock.scripps.edu/wp-content/uploads/sites/31/2019/03/AD4.1_bound.dat

In [17]:
%%bash
cd /home/ssm-user/project/autodock
~/apps/mgltools/bin/pythonsh ~/apps/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_gpf4.py \
    -l a1aac.pdbqt \
    -r 8vb5_prot.pdbqt \
    -y \
    -p ligand_types='H,HD,A,C,N,NA,NS,OA,OS,F,Cl,Br,S,SA' \
    -p npts='80,80,80'

setting PYTHONHOME environment
setting ligand_types: newvalue= H HD A C N NA NS OA OS F Cl Br S SA


In [18]:
%%bash
cd /home/ssm-user/project/autodock
/home/ssm-user/miniforge3/envs/docking/bin/autogrid4 -p 8vb5_prot.gpf -l 8vb5_prot.glg
ls -la *.fld *.map
cd ../

-rw-r--r-- 1 ssm-user ssm-user 4114935 Aug 18 15:15 8vb5_prot.A.map
-rw-r--r-- 1 ssm-user ssm-user 4286075 Aug 18 15:15 8vb5_prot.Br.map
-rw-r--r-- 1 ssm-user ssm-user 4123625 Aug 18 15:15 8vb5_prot.C.map
-rw-r--r-- 1 ssm-user ssm-user 4201643 Aug 18 15:15 8vb5_prot.Cl.map
-rw-r--r-- 1 ssm-user ssm-user 3891147 Aug 18 15:15 8vb5_prot.F.map
-rw-r--r-- 1 ssm-user ssm-user 3646053 Aug 18 15:15 8vb5_prot.H.map
-rw-r--r-- 1 ssm-user ssm-user 3696438 Aug 18 15:15 8vb5_prot.HD.map
-rw-r--r-- 1 ssm-user ssm-user 4034773 Aug 18 15:15 8vb5_prot.N.map
-rw-r--r-- 1 ssm-user ssm-user 4041111 Aug 18 15:15 8vb5_prot.NA.map
-rw-r--r-- 1 ssm-user ssm-user 4041289 Aug 18 15:15 8vb5_prot.NS.map
-rw-r--r-- 1 ssm-user ssm-user 4019460 Aug 18 15:15 8vb5_prot.OA.map
-rw-r--r-- 1 ssm-user ssm-user 4019460 Aug 18 15:15 8vb5_prot.OS.map
-rw-r--r-- 1 ssm-user ssm-user 4158929 Aug 18 15:15 8vb5_prot.S.map
-rw-r--r-- 1 ssm-user ssm-user 4175413 Aug 18 15:15 8vb5_prot.SA.map
-rw-r--r-- 1 ssm-user ssm-user 3182085 A

### 3. Get pdbqt from potent smiles(Enamine)

In [20]:
!mkdir -p /home/ssm-user/project/autodock/potent
!mkdir -p /home/ssm-user/project/autodock/potent_pdbqt

### [Bash] Run "03_make_smi_from_csv.py"
---
python 03_make_smi_from_csv.py /home/ssm-user/project/smiles_batch_2_20029.csv /home/ssm-user/project/autodock/potent

In [21]:
potent_path = Path("/home/ssm-user/project/autodock/potent")
smi_files = list(potent_path.glob("*.smi"))
print(f"Created .smi: {len(smi_files)}")

Created .smi: 20029


In [23]:
cpu_count = os.cpu_count()
print(f"avaliable CPU cores: {cpu_count}")

avaliable CPU cores: 8


### [Bash] Run "04_prepare_pdbqt_parallel.sh"
---
chmod +x 04_prepare_pdbqt_parallel.sh

export JOBS=8

./04_prepare_pdbqt_parallel.sh

In [57]:
potent_pdbqt_path = Path("/home/ssm-user/project/autodock/potent_pdbqt")
pdbqt_files = list(potent_pdbqt_path.glob("*.pdbqt"))
print(f"Created .pdbqt: {len(pdbqt_files)}")

Created .pdbqt: 19900


### 4. Making a batch file
---------
https://github.com/ccsb-scripps/AutoDock-GPU

In [31]:
list_path = Path("/home/ssm-user/project/autodock/ligand_list.txt")
with open(list_path, "w") as f:
    f.write("./8vb5_prot.maps.fld\n")
print(f"{list_path}")

/home/ssm-user/project/autodock/ligand_list.txt


### 5. Run AutoDock

In [33]:
with open('/home/ssm-user/project/autodock/ligand_list.txt', 'w') as fout:
    fout.write('./8vb5_prot.maps.fld \n') 
    files = glob.glob('./potent_pdbqt/*.pdbqt')
    
    for filename in files:
        fout.write(os.path.abspath(filename) + '\n')
        fout.write(filename.split('/')[-1].split('.')[0] + '\n')

In [46]:
# rm -f *.dlg *.xml

In [ ]:
%%bash
cd /home/ssm-user/project/autodock
# rm -f *.dlg *.xml
 
~/apps/AutoDock-GPU/bin/autodock_gpu_128wi -B ./ligand_list.txt | tee gpu_out

### 6. Analyze gpu_out

In [66]:
def get_min_affinity(dlg_file):
    with open(dlg_file, 'r') as f:
        lines = f.readlines()

    affinity_list = []
    for line in lines:
        if 'Estimated Free Energy of Binding' in line:
            try:
                affinity = float(line.strip().split()[-3])
                affinity_list.append(affinity)
            except (ValueError, IndexError):
                continue
    
    if affinity_list:
        return min(affinity_list)
    else:
        return None

In [67]:
dlg_files = glob.glob('/home/ssm-user/project/autodock/*.dlg') 
lig_and_energy = []

print("Processing dlg files...")
for f in dlg_files:
    min_e = get_min_affinity(f)
    if min_e is not None: 
        lig_name = os.path.basename(f).split('.')[0]  
        lig_and_energy.append((lig_name, min_e))

lig_and_energy.sort(key=lambda x: x[1])

print(f"\n=== TOP LIGANDS (Total: {len(lig_and_energy)}) ===")
print(f"{'Rank':<5} {'Ligand':<25} {'Binding Affinity (kcal/mol)':<25}")
print("-" * 60)

# for rank, (ligand, energy) in enumerate(lig_and_energy, 1):
#     print(f"{rank:<5} {ligand:<25} {energy:<25.2f}")

print(f"\n=== TOP 20 ===")
for rank, (ligand, energy) in enumerate(lig_and_energy[:20], 1):
    print(f"{rank}. {ligand}: {energy:.2f} kcal/mol")

Processing dlg files...

=== TOP LIGANDS (Total: 20025) ===
Rank  Ligand                    Binding Affinity (kcal/mol)
------------------------------------------------------------

=== TOP 20 ===
1. Z2989433123: -5.25 kcal/mol
2. Z2171583301: -5.23 kcal/mol
3. Z1385377331: -5.21 kcal/mol
4. Z1175821707: -5.10 kcal/mol
5. Z2116831182: -5.07 kcal/mol
6. Z2809371707: -5.06 kcal/mol
7. Z3174537988: -5.00 kcal/mol
8. Z1401321648: -4.92 kcal/mol
9. Z85101640: -4.89 kcal/mol
10. Z1217532529: -4.88 kcal/mol
11. Z1252919934: -4.86 kcal/mol
12. Z2839199517: -4.85 kcal/mol
13. Z1497483528: -4.83 kcal/mol
14. Z2171241571: -4.83 kcal/mol
15. Z1334095596: -4.80 kcal/mol
16. Z2437406281: -4.80 kcal/mol
17. Z1024518512: -4.78 kcal/mol
18. Z296875264: -4.77 kcal/mol
19. Z1251121499: -4.77 kcal/mol
20. Z318397580: -4.77 kcal/mol


In [68]:
out = "/home/ssm-user/project/autodock/potent/name2smi_from_smi.txt"
os.makedirs(os.path.dirname(out), exist_ok=True)

with open(out, "w") as f:
    for p in sorted(glob.glob("/home/ssm-user/project/autodock/potent/*.smi")):
        name = os.path.basename(p).rsplit(".", 1)[0]
        with open(p, "r") as g:
            smi = g.read().strip()
        if smi:
            f.write(f"{name}\t{smi}\n")

In [70]:
MAP_PATH="/home/ssm-user/project/autodock/potent/name2smi_from_smi.txt"
name2smi={}
if os.path.exists(MAP_PATH):
    with open(MAP_PATH) as fh:
        for line in fh:
            k,v=line.strip().split('\t',1)
            name2smi[k]=v

out="/home/ssm-user/project/autodock/ligand_and_affinity.txt"
with open(out,"w") as fout:
    fout.write('Rank\tLigand\tSMILES\tBinding_Affinity(kcal/mol)\n')
    for rank,(lig,ene) in enumerate(lig_and_energy,1):
        smi=name2smi.get(lig,'N/A')
        fout.write(f"{rank}\t{lig}\t{smi}\t{ene:10.2f}\n")
print("Wrote", out)

Wrote /home/ssm-user/project/ligand_and_affinity.txt


In [26]:
!head /home/ssm-user/project/autodock/potent/name2smi_from_smi.txt

Z1000268726	O=C(NC=1C=CC=C(C1)C(=O)NC=2C=CC=CN2)C=3C=CSC3
Z1000270702	CC=1N=C(SC1C(=O)NC=2C=CC=C(C2)C(=O)NC=3C=CN=CC3)C(C)(C)C
Z1000825674	CC1=C(N=NN1C=2C=CC(C)=CC2)C(=O)NCC3CCN(C)CC3
Z1001361900	CC1=CC=C(O1)C(CNC(=O)CN2C=NC=3C=CC=CC32)N4CCCC4
Z1001603724	CC1=CSC(=N1)C=2C=CC(=CC2)NC(=O)C=3C=C(NN3)C4=CC=CO4
Z1001778380	COCCN1C=CC(=N1)NC(=O)CCN2C=CC=3C=CC=CC32
Z1002336518	COC=1C=CC=C2C1OCCC2NC=3N=CN=C4C3C=NN4C
Z1002427608	O=C(CNC=1C=CC=2SC=NC2C1)NC3CCCCC3
Z1002777646	CC(C=1C=CC=C(Cl)C1)N2CCN(CC2)C(=O)C3=CC=4C=CC=CC4N3
Z1003052272	CCCN(CCOC=1C=CC=CC1Cl)C2CCN(CC2)C(=O)C=3C=CC=CC3


In [ ]:
# !obabel -ad -ipdbqt ./virtual_screening/Acetohexamide.dlg  -opdbqt -O ./virtual_screening/Acetohexamide.docked.pdbqt

### 7. smiles with salt (preprocessing mistaken)

In [49]:
csv_file = "/home/ssm-user/project/smiles_batch_2_20029.csv"
txt_file = "/home/ssm-user/project/autodock/ligand_and_affinity.txt"

df_csv = pd.read_csv(csv_file)
csv_smiles = set(df_csv["SMILES"].astype(str))

df_txt = pd.read_csv(txt_file, sep="\t", header=0)
txt_smiles = set(df_txt["SMILES"].astype(str))

print("input smiles:", len(csv_smiles))
print("output smiles:", len(txt_smiles) + 3)         # three of them have an error during autodock
print("missing:", len(csv_smiles - txt_smiles) - 3)

missing_df = df_csv[df_csv["SMILES"].isin(csv_smiles - txt_smiles)]
missing_df.to_csv("/home/ssm-user/project/missing.csv", index=False)

input smiles: 20029
output smiles: 19900
missing: 129


In [46]:
dot_count = missing_df["SMILES"].str.contains("\.").sum()
print(dot_count)

128


In [48]:
dotted_df = missing_df[missing_df["SMILES"].str.contains("\.")].copy()
dotted_df["SMILES"] = dotted_df["SMILES"].str.split("\.").str[1]
dotted_df.to_csv("/home/ssm-user/project/dotted_batch_2.csv", index=False)

[Bash]
---
python 03_make_smi_from_csv.py /home/ssm-user/project/dotted_batch_2.csv /home/ssm-user/project/autodock/potent

chmod +x 04_dot_prepare_pdbqt_parallel.sh

export JOBS=8

./04_dot_prepare_pdbqt_parallel.sh

In [61]:
!ls /home/ssm-user/project/autodock/dot_potent_pdbqt/*.pdbqt 2>/dev/null | wc -l

128


In [62]:
list_path = Path("/home/ssm-user/project/autodock/dot_ligand_list.txt")
with open(list_path, "w") as f:
    f.write("./8vb5_prot.maps.fld\n")
print(f"{list_path}")

/home/ssm-user/project/autodock/dot_ligand_list.txt


In [63]:
with open('/home/ssm-user/project/autodock/dot_ligand_list.txt', 'w') as fout:
    fout.write('./8vb5_prot.maps.fld \n') 
    files = glob.glob('./dot_potent_pdbqt/*.pdbqt')
    
    for filename in files:
        fout.write(os.path.abspath(filename) + '\n')
        fout.write(filename.split('/')[-1].split('.')[0] + '\n')

In [ ]:
%%bash
cd /home/ssm-user/project/autodock
# rm -f *.dlg *.xml
~/apps/AutoDock-GPU/bin/autodock_gpu_128wi -B ./dot_ligand_list.txt | tee gpu_out

Go back to 6. Analyze gpu_out